In [1]:
from investment_agents.graph.investment_graph import build_investment_graph

from investment_agents.snapshots.technical import TechnicalSnapshotService
from investment_agents.snapshots.chip import ChipSnapshotService
from investment_agents.snapshots.regime import MarketRegimeSnapshotService

# 依照你目前實際的 Repository / Client import 路徑調整
from investment_agents.data.clients.finmind import FinMindClient
from investment_agents.data.repositories.price import PriceRepository
from investment_agents.data.repositories.dividend import DividendRepository
from investment_agents.data.repositories.institutional import InstitutionalRepository
from investment_agents.data.repositories.market_regime import MarketRegimeRepository

from investment_agents.features.price_adjustment import PriceAdjustmentService
from investment_agents.features.technical import TechnicalFeatureService
from investment_agents.features.chip import ChipFeatureService
from investment_agents.features.regime import RegimeFeatureService


# ============================================================
# Config
# ============================================================

TICKER = "2330"
COMPANY_NAME = "台積電"
AS_OF_DATE = "2026-09-18"


# ============================================================
# Data Client
# ============================================================

client = FinMindClient()


# ============================================================
# Repositories
# ============================================================

price_repository = PriceRepository(client)
dividend_repository = DividendRepository(client)
institutional_repository = InstitutionalRepository(client)
regime_repository = MarketRegimeRepository(client)


# ============================================================
# Feature Services
# ============================================================

price_adjustment_service = PriceAdjustmentService()

technical_feature_service = TechnicalFeatureService()

chip_feature_service = ChipFeatureService()

regime_feature_service = RegimeFeatureService()


# ============================================================
# Snapshot Services
# ============================================================

technical_snapshot_service = TechnicalSnapshotService(
    price_repo=price_repository,
    dividend_repo=dividend_repository,
    adjustment_service=price_adjustment_service,
    technical_service=technical_feature_service,
)

chip_snapshot_service = ChipSnapshotService(
    institutional_repo=institutional_repository,
    price_repo=price_repository,
    chip_service=chip_feature_service,
)

regime_snapshot_service = MarketRegimeSnapshotService(
    repository=regime_repository,
    feature_service=regime_feature_service,
)


# ============================================================
# Build point-in-time snapshots
# ============================================================

technical_data = technical_snapshot_service.get_snapshot(
    ticker=TICKER,
    as_of_date=AS_OF_DATE,
)

chip_data = chip_snapshot_service.get_snapshot(
    ticker=TICKER,
    as_of_date=AS_OF_DATE,
)

regime_data = regime_snapshot_service.get_snapshot(
    as_of_date=AS_OF_DATE,
)


print("\n========== Technical Snapshot ==========")
print(technical_data)

print("\n========== Chip Snapshot ==========")
print(chip_data)

print("\n========== Regime Snapshot ==========")
print(regime_data)


# ============================================================
# Initial LangGraph State
# ============================================================

initial_state = {
    "ticker": TICKER,
    "company_name": COMPANY_NAME,

    "technical_data": technical_data,
    "chip_data": chip_data,
    "regime_data": regime_data,
}


# ============================================================
# Build + Run Graph
# ============================================================

graph = build_investment_graph()

result = graph.invoke(initial_state)


# ============================================================
# Results
# ============================================================

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)

technical_report = result["technical_report"]
chip_report = result["chip_report"]
regime_report = result["regime_report"]
pm_report = result["pm_report"]

print("\n[Technical]")
print(f"Score:  {technical_report.score}")
print(f"Reason: {technical_report.reason}")

print("\n[Chip]")
print(f"Score:  {chip_report.score}")
print(f"Reason: {chip_report.reason}")

print("\n[Regime]")
print(f"Regime:     {regime_report.regime}")
print(f"Risk Score: {regime_report.risk_score}")
print(f"Confidence: {regime_report.confidence}")
print(f"Summary:    {regime_report.summary}")

print("\n[Portfolio Manager]")
print(f"Final Score: {pm_report.final_score}")
print(f"Conviction:  {pm_report.conviction}")
print(f"Reason:      {pm_report.reason}")

2026-09-21 01:07:50.981 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-09-21 01:07:51.029 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-09-21 01:07:51.037 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330
2026-09-21 01:07:51.287 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockDividendResult, data_id: 2330
2026-09-21 01:07:51.371 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330
2026-09-21 01:07:51.441 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2330
2026-09-21 01:07:51.549 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-21 01:07:51.607 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors,


========== Technical Snapshot ==========
{'roc_5d': 2.3755914785524634, 'roc_10d': 2.3755914785524634, 'roc_20d': 2.3755914785524634, 'roc_1m': 3.884284405604821, 'roc_3m': -1.703117345294236, 'roc_6m': 33.721352316293185, 'roc_12m': 115.5345143730143, 'bollinger_z': 1.61555898448717, 'rsi': 58.179469100051726, 'macd': 10.134524419488116, 'macd_signal': 10.090813393036596, 'macd_hist': 0.04371102645152014, 'stochastic_k': 39.044335762555846, 'stochastic_d': 19.786372337796085, 'stochastic_j': 77.56026261207538}

========== Chip Snapshot ==========
{'foreign_flow_ratio_1d': 14.768782135329426, 'foreign_flow_ratio_5d': -11.42463030058623, 'foreign_flow_ratio_20d': -1.0174058979023712, 'foreign_buy_days_20d': 9.0, 'foreign_streak': 2.0, 'trust_flow_ratio_1d': 1.1698937472635693, 'trust_flow_ratio_5d': 0.9462433319172925, 'trust_flow_ratio_20d': -0.6151055780971385, 'trust_buy_days_20d': 8.0, 'trust_streak': 1.0, 'dealer_flow_ratio_1d': 2.3929583694767143, 'dealer_flow_ratio_5d': 1.400090

In [2]:
import pandas as pd

from investment_agents.data.clients.finmind import FinMindClient
from investment_agents.data.repositories.price import PriceRepository
from investment_agents.data.repositories.dividend import DividendRepository

from investment_agents.features.price_adjustment import PriceAdjustmentService
from investment_agents.features.technical import TechnicalFeatureService

In [3]:
client = FinMindClient()

price_repo = PriceRepository(client)
dividend_repo = DividendRepository(client)

adjustment_service = PriceAdjustmentService()
technical_service = TechnicalFeatureService()

In [4]:
ticker = "2330"
as_of_date = "2026-09-18"

start_date = "2025-06-01"

In [6]:
price_df = price_repo.get_history(
    ticker=ticker,
    start_date=start_date,
    end_date=as_of_date,
)

print(price_df.tail(30).to_string(index=False))

2026-09-21 01:12:09.643 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330


      date ticker   open   high    low  close   volume  trading_money  turnover
2026-08-10   2330 2390.0 2410.0 2380.0 2380.0 21498241    51385081395     67328
2026-08-11   2330 2390.0 2405.0 2375.0 2395.0 18247582    43667182348     50065
2026-08-12   2330 2405.0 2415.0 2390.0 2415.0 19448153    46798062170     63548
2026-08-13   2330 2440.0 2445.0 2425.0 2435.0 26233385    63854331391     73397
2026-08-14   2330 2435.0 2440.0 2395.0 2395.0 21162682    51159731253    105889
2026-08-17   2330 2410.0 2420.0 2390.0 2400.0 13482456    32423014050     56197
2026-08-18   2330 2415.0 2415.0 2375.0 2380.0 19997213    47742904360    102735
2026-08-19   2330 2340.0 2355.0 2335.0 2350.0 23618612    55394691541    193704
2026-08-20   2330 2365.0 2375.0 2350.0 2375.0 16967737    40117420293     46361
2026-08-21   2330 2375.0 2410.0 2365.0 2410.0 18922480    45275662448     59539
2026-08-24   2330 2410.0 2410.0 2375.0 2375.0 13073210    31234578255     84694
2026-08-25   2330 2355.0 2400.0 2350.0 2

In [7]:
print("shape:", price_df.shape)
print("latest date:", price_df["date"].max())
print("duplicate dates:", price_df["date"].duplicated().sum())

print(
    price_df[["date", "close"]]
    .tail(30)
    .to_string(index=False)
)

shape: (321, 9)
latest date: 2026-09-18 00:00:00
duplicate dates: 0
      date  close
2026-08-10 2380.0
2026-08-11 2395.0
2026-08-12 2415.0
2026-08-13 2435.0
2026-08-14 2395.0
2026-08-17 2400.0
2026-08-18 2380.0
2026-08-19 2350.0
2026-08-20 2375.0
2026-08-21 2410.0
2026-08-24 2375.0
2026-08-25 2400.0
2026-08-26 2415.0
2026-08-27 2410.0
2026-08-28 2420.0
2026-08-31 2405.0
2026-09-01 2440.0
2026-09-02 2385.0
2026-09-03 2390.0
2026-09-04 2410.0
2026-09-07 2460.0
2026-09-08 2470.0
2026-09-09 2465.0
2026-09-10 2450.0
2026-09-11 2410.0
2026-09-14 2380.0
2026-09-15 2385.0
2026-09-16 2380.0
2026-09-17 2425.0
2026-09-18 2460.0


In [8]:
dividend_df = dividend_repo.get_history(
    ticker=ticker,
    start_date=start_date,
    end_date=as_of_date,
)

print(dividend_df.tail(20).to_string(index=False))

2026-09-21 01:12:41.646 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockDividendResult, data_id: 2330


      date ticker  before_price  after_price
2025-06-12   2330        1065.0      1060.49
2025-09-16   2330        1255.0      1249.99
2025-12-11   2330        1505.0      1499.99
2026-03-17   2330        1845.0      1838.99
2026-06-11   2330        2255.0      2248.99
2026-09-16   2330        2385.0      2377.99


In [10]:
adjusted_df = adjustment_service.transform(
    price_df=price_df,
    dividend_df=dividend_df,
)

print(
    adjusted_df[
        ["date", "close", "adj_close"]
    ]
    .tail(30)
    .to_string(index=False)
)

      date  close   adj_close
2026-08-10 2380.0 2373.004696
2026-08-11 2395.0 2387.960608
2026-08-12 2415.0 2407.901824
2026-08-13 2435.0 2427.843040
2026-08-14 2395.0 2387.960608
2026-08-17 2400.0 2392.945912
2026-08-18 2380.0 2373.004696
2026-08-19 2350.0 2343.092872
2026-08-20 2375.0 2368.019392
2026-08-21 2410.0 2402.916520
2026-08-24 2375.0 2368.019392
2026-08-25 2400.0 2392.945912
2026-08-26 2415.0 2407.901824
2026-08-27 2410.0 2402.916520
2026-08-28 2420.0 2412.887128
2026-08-31 2405.0 2397.931216
2026-09-01 2440.0 2432.828344
2026-09-02 2385.0 2377.990000
2026-09-03 2390.0 2382.975304
2026-09-04 2410.0 2402.916520
2026-09-07 2460.0 2452.769560
2026-09-08 2470.0 2462.740168
2026-09-09 2465.0 2457.754864
2026-09-10 2450.0 2442.798952
2026-09-11 2410.0 2402.916520
2026-09-14 2380.0 2373.004696
2026-09-15 2385.0 2377.990000
2026-09-16 2380.0 2380.000000
2026-09-17 2425.0 2425.000000
2026-09-18 2460.0 2460.000000


In [11]:
df = adjusted_df.sort_values("date").reset_index(drop=True)

latest = df.iloc[-1]

for n in [5, 10, 20]:
    past = df.iloc[-1 - n]

    roc = (
        latest["adj_close"] / past["adj_close"] - 1
    ) * 100

    print(
        f"{n}D | "
        f"latest={latest['date'].date()} "
        f"latest_price={latest['adj_close']:.4f} | "
        f"past={past['date'].date()} "
        f"past_price={past['adj_close']:.4f} | "
        f"RoC={roc:.6f}%"
    )

5D | latest=2026-09-18 latest_price=2460.0000 | past=2026-09-11 past_price=2402.9165 | RoC=2.375591%
10D | latest=2026-09-18 latest_price=2460.0000 | past=2026-09-04 past_price=2402.9165 | RoC=2.375591%
20D | latest=2026-09-18 latest_price=2460.0000 | past=2026-08-21 past_price=2402.9165 | RoC=2.375591%


In [12]:
print(dividend_df.tail(10).to_string(index=False))

event = dividend_df[
    dividend_df["date"] == pd.Timestamp("2026-09-16")
]

print(event)

if not event.empty:
    before = event.iloc[0]["before_price"]
    after = event.iloc[0]["after_price"]

    print("before_price:", before)
    print("after_price :", after)
    print("factor      :", after / before)

      date ticker  before_price  after_price
2025-06-12   2330        1065.0      1060.49
2025-09-16   2330        1255.0      1249.99
2025-12-11   2330        1505.0      1499.99
2026-03-17   2330        1845.0      1838.99
2026-06-11   2330        2255.0      2248.99
2026-09-16   2330        2385.0      2377.99
        date ticker  before_price  after_price
5 2026-09-16   2330        2385.0      2377.99
before_price: 2385.0
after_price : 2377.99
factor      : 0.9970607966457022
